# HTTP Requests and APIs

> **The big idea:** most useful data does not live inside your program - it lives on another computer somewhere on the internet (a *server*). To get it, your program sends a **request**, the request travels across the internet, and the server sends back a **response**. That conversation is called **HTTP**.

- Understand client, server, request, and response
- Send HTTP requests with `httpx`
- Use GET, POST, PUT, PATCH, and DELETE
- Work with paths, query parameters, headers, and JSON
- Check status codes and handle errors
- Connect API responses to Pydantic models

We use **DummyJSON** (`https://dummyjson.com`) - a free, real API running on the internet that serves shopping data: products, categories, reviews, carts. You send real requests; a real machine replies. You learn GET, the address (URL), status codes, JSON, query filters, POST, headers, timeouts, and handling errors.

> **Setup:** Run `pip install httpx pydantic` if needed. The API examples require an internet connection.

## What is an API?

Think of an API as a way for one program to ask another program for data or to request an action.

- **Client:** our Python program.
- **Server:** the remote program that handles the request.
- **Request:** what we send (method, URL, optional headers and body).
- **Response:** what comes back (status code, headers and body).
- **JSON:** a common format for exchanging structured data.

Our Python code is the client. We'll use DummyJSON as a practice server.

**The shape of every HTTP call we write today:**
```
httpx.get(ADDRESS, params={...}, timeout=10.0)
        |          |              |
        |          |              +-- how many SECONDS to wait before giving up
        |          +-- optional extra options added after a '?' in the address
        +-- the web address (URL) of what you want
```
Every call returns a **response** object. From it you read `.status_code` (a number: did it work?) and `.json()` (the actual data, as a Python dict). Those three - the call, the status, the data - are the whole skill.


In [1]:
import httpx

BASE_URL = "https://dummyjson.com"

# Let's check we can actually reach the server before we start.
# httpx.get(URL) sends a GET request (a 'please give me data' request) to that web address.
#   timeout=10.0  ->  wait at most 10 SECONDS for a reply. If the server hasn't answered by then,
#                     httpx stops waiting and raises an error instead of freezing forever.
#                     (Without a timeout, a stuck server could freeze your program indefinitely.)
r = httpx.get(f'{BASE_URL}/test', timeout=10.0)

print('Reached the server. Status:', r.status_code)   # .status_code is a number telling us how it went (200 = OK)
print('Server says:', r.json())                       # .json() turns the reply text into a Python dictionary

Reached the server. Status: 200
Server says: {'status': 'ok', 'method': 'GET'}


## The Four Things You Can Ask a Server to Do

Almost every API call is one of four **verbs** - the *action* you want. You have met GET and POST; now we complete the set.

| Verb | Plain meaning | Everyday picture | Example |
|------|---------------|------------------|---------|
| **GET** | read / fetch something | looking something up | `GET /products/1` |
| **POST** | create something new | filling in a brand-new form | `POST /products/add` |
| **PUT** | update something that exists | editing a form you already submitted | `PUT /products/1` |
| **DELETE** | remove something | throwing something away | `DELETE /products/1` |

In `httpx` each one is just a method with the same shape you already know: `httpx.get(...)`, `httpx.post(...)`, `httpx.put(...)`, `httpx.delete(...)`.


## 1. First request: GET a product

`GET` asks the server to return data. `httpx.get()` sends the request; `.status_code` tells us the result; `.json()` converts a JSON response into Python data.

In [2]:
import json

# Send a GET request asking for the list of products.
response = httpx.get(f'{BASE_URL}/products', timeout=10.0)

# .status_code is a 3-digit number the server sends back to say how the request went.
print('Status code:', response.status_code)

# The reply arrives as plain TEXT. .json() converts that text into a real Python dictionary
# so we can pull values out of it with square brackets, just like any dict.
data = response.json()
print(json.dumps(data, indent=2))

print('Total products on the server:', data['total'])    # 'total' = how many products exist in all
print('How many we got back:', len(data['products']))    # the server sends them in pages (30 at a time by default)
print('First product:', data['products'][0]['title'])    # data['products'] is a list; [0] = the first one

Status code: 200
{
  "products": [
    {
      "id": 1,
      "title": "Essence Mascara Lash Princess",
      "description": "The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.",
      "category": "beauty",
      "price": 9.99,
      "discountPercentage": 10.48,
      "rating": 2.56,
      "stock": 99,
      "tags": [
        "beauty",
        "mascara"
      ],
      "brand": "Essence",
      "sku": "BEA-ESS-ESS-001",
      "weight": 4,
      "dimensions": {
        "width": 15.14,
        "height": 13.08,
        "depth": 22.99
      },
      "warrantyInformation": "1 week warranty",
      "shippingInformation": "Ships in 3-5 business days",
      "availabilityStatus": "In Stock",
      "reviews": [
        {
          "rating": 3,
          "comment": "Would not recommend!",
          "date": "2025-04-30T09:41:02.053Z",
          "reviewerName": "Eleanor Col

> The server returned `total`, `skip`, and `limit` too. By default it sends 30 products at a time (this is called *pagination* - sending big lists in pages).

## 2. Read a list and use query parameters

A path identifies the resource. Query parameters let us filter or control the results. Use `params=` instead of manually building a query string.

In [5]:
response = httpx.get(f"{BASE_URL}/products", params={"limit": 5}, timeout=10.0)

# response.raise_for_status()
data = response.json()

print("Products returned:", len(data["products"]))
for item in data["products"]:
    print(item["id"], item["title"], "-", item["price"])

Products returned: 5
1 Essence Mascara Lash Princess - 9.99
2 Eyeshadow Palette with Mirror - 19.99
3 Powder Canister - 14.99
4 Red Lipstick - 12.99
5 Red Nail Polish - 8.99


For example, `params={"limit": 5}` becomes `?limit=5` in the URL. `raise_for_status()` raises an exception if the server returns an unsuccessful HTTP status, such as 404 or 500.

In [3]:
# Search products by a phrase.
response = httpx.get(f"{BASE_URL}/products/search", params={"q": "pen"}, timeout=10.0)
response.raise_for_status()

for item in response.json()["products"]:
    print(item["title"])

Samsung Galaxy Tab S8 Plus Grey


In [9]:
# Searching is just another query option. The '/products/search' endpoint looks for matching text.
#   params={'q': 'phone'}  ->  becomes  ?q=phone  ('q' is short for 'query', the thing to search for)
found = httpx.get(f'{BASE_URL}/products/search', params={'q': 'phone'}, timeout=10.0).json()

print(f"Found {found['total']} products matching 'phone'. First few:")
for p in found['products'][:5]:          # [:5] = just the first 5 from the list
    print('  ', p['title'])

Found 23 products matching 'phone'. First few:
   Apple AirPods Max Silver
   Apple iPhone Charger
   Apple MagSafe Battery Pack
   Beats Flex Wireless Earphones
   iPhone 12 Silicone Case with MagSafe Plum


## 3. Common status codes

| Code | Meaning | Typical interpretation |
|---|---|---|
| `200` | OK | Request succeeded |
| `201` | Created | A resource was created |
| `400` | Bad Request | The request data is invalid |
| `401` | Unauthorized | Authentication is missing or invalid |
| `403` | Forbidden | You are not allowed to do this |
| `404` | Not Found | The resource does not exist |
| `429` | Too Many Requests | Rate limit reached |
| `500` | Internal Server Error | The server failed |

Don't assume every successful request returns exactly `200`. For example, creation may return `201`.

Memory aid: **2xx = all good**, **4xx = your fault** (asked wrong), **5xx = their fault** (server broke).

In [7]:
# Try requesting a product ID that probably does not exist.
response = httpx.get(f"{BASE_URL}/products/999999", timeout=10.0)

print("Status:", response.status_code)
if response.status_code == 200:
    print(response.json())
elif response.status_code == 404:
    print("Product does not exist.")
else:
    print("Request returned another status:", response.status_code)

Status: 404
Product does not exist.


In [5]:
# Shortcut: instead of checking by hand, .raise_for_status() will AUTOMATICALLY raise an
# error if the status is a failure code (4xx or 5xx). You can then catch it with try/except.
try:
    httpx.get(f'{BASE_URL}/products/99999', timeout=10.0).raise_for_status()
except httpx.HTTPStatusError as e:       # this 'except' runs only if the status was a failure
    print('raise_for_status caught:', e.response.status_code)

raise_for_status caught: 404


## 4 - Explore the Whole Store

Now that you can call endpoints, look at the other kinds of data this real server offers.

In [10]:
# Ask the server for the list of all category NAMES.
# This endpoint returns a plain list of strings (not a dict with 'products' inside).
cats = httpx.get(f'{BASE_URL}/products/category-list', timeout=10.0).json()

print('There are', len(cats), 'categories. Some of them:')
print('  ', cats[:10])                   # [:10] = first 10 category names

There are 24 categories. Some of them:
   ['beauty', 'fragrances', 'furniture', 'groceries', 'home-decoration', 'kitchen-accessories', 'laptops', 'mens-shirts', 'mens-shoes', 'mens-watches']


In [12]:
# Get every product in ONE category. Here the category name is part of the PATH:
#   /products/category/smartphones  ->  'all products in the smartphones category'
smartphones = httpx.get(f'{BASE_URL}/products/category/smartphones', timeout=10.0).json()

print(f"Smartphones category has {smartphones['total']} products:")
for p in smartphones['products'][:5]:    # show just the first 5
    print(f"  {p['title']:<28} ${p['price']}")

Smartphones category has 16 products:
  iPhone 5s                    $199.99
  iPhone 6                     $299.99
  iPhone 13 Pro                $1099.99
  iPhone X                     $899.99
  Oppo A57                     $249.99


In [20]:
# On this server, reviews are not a separate endpoint - they come INSIDE each product,
# under the 'reviews' key. So we fetch the product, then read its 'reviews' list.
product = httpx.get(f'{BASE_URL}/products/8', timeout=10.0).json()

print('Reviews for:', product['title'])
for rev in product['reviews']:           # each 'rev' is one review dict
    # build a star string: filled stars for the rating, empty stars for the rest (out of 5)
    stars = '★' * rev['rating'] + '☆' * (5 - rev['rating'])
    print(f"  {stars}  {rev['reviewerName']}: {rev['comment']}")

Reviews for: Dior J'adore
  ★★★★★  Nicholas Bailey: Great value for money!
  ★★★★☆  Penelope Harper: Great value for money!
  ★★★★☆  Emma Miller: Great product!


In [24]:
# A 'cart' here works like an order: it has a list of products, each with a quantity and a total.
cart = httpx.get(f'{BASE_URL}/carts/1', timeout=10.0).json()   # /carts/1 = the cart whose id is 1

print('Cart', cart['id'], '- total $' + str(cart['total']), 'for', cart['totalProducts'], 'products')
for item in cart['products'][:4]:        # show the first 4 items in the cart
    print(f"  {item['title']:<25} x {item['quantity']}  = ${item['total']}")

Cart 1 - total $13037.88 for 4 products
  Blue Frock                x 4  = $119.96
  Generic Motorcycle        x 3  = $11999.97
  iPhone 6                  x 3  = $899.97
  Baseball Ball             x 2  = $17.98


## 5. Send data with POST

`POST` commonly sends data to create a resource. Put the data in a Python dictionary and pass it using `json=`.

In [25]:
new_product = {
    "title": "Voila Notebook",
    "price": 9.50,
    "category": "stationery"
}

response = httpx.post(f"{BASE_URL}/products/add", json=new_product, timeout=10.0)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 201
Response: {'id': 195, 'title': 'Voila Notebook', 'price': 9.5, 'category': 'stationery'}


DummyJSON simulates adding the product and returns a response, but the record is not permanently stored. In a real API, follow the service's documentation to understand its behavior and permissions.

**Remember:** `json=payload` asks `httpx` to encode the dictionary as JSON and set the appropriate content type.

## 6. Update with PUT or PATCH

- `PUT` commonly replaces a resource representation.
- `PATCH` commonly changes selected fields.


Exact behavior depends on the API's documentation.

In [26]:
# Demo update (simulated by DummyJSON).
response = httpx.patch(f"{BASE_URL}/products/1", json={"price": 99.99}, timeout=10.0)

print("PATCH status:", response.status_code)
updated = response.json()
print('Product :', updated['title'])
print('New price:', updated['price'])

PATCH status: 200
Product : Essence Mascara Lash Princess
New price: 99.99


### DELETE - remove something
**DELETE** asks the server to remove a thing. DummyJSON replies with the item it 'deleted', plus two extra labels - `isDeleted` and `deletedOn` - so you can confirm it worked.

> Again, this is **simulated** - the product is not really removed from the server. Safe to run freely.

In [27]:
# Demo delete (simulated by DummyJSON).
response = httpx.delete(f"{BASE_URL}/products/1", timeout=10.0)

print('Status:', response.status_code)          # 200 = the delete request was handled
deleted = response.json()
print('Deleted product:', deleted['title'])
print('isDeleted flag :', deleted['isDeleted'])     # the server adds this flag to confirm: True
print('Deleted at     :', deleted['deletedOn'])     # ...and a timestamp of when it 'happened'

Status: 200
Deleted product: Essence Mascara Lash Princess
isDeleted flag : True
Deleted at     : 2026-11-23T17:18:30.236Z


## 7. Headers and authentication

Headers carry extra information with a request. For example, `Accept` tells the server what response format the client prefers. Many APIs also require an API key or bearer token in an `Authorization` header.

Never hard-code real secrets in a notebook or commit them to source control.

In [28]:
response = httpx.get(f"{BASE_URL}/products/1", headers={"Accept": "application/json"}, timeout=10.0)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))

Status: 200
Content-Type: application/json; charset=utf-8


## 8. Handle network and HTTP errors

Network calls can fail because of timeouts, connection problems, or unsuccessful HTTP responses. Catch errors you can reasonably handle instead of assuming every call succeeds.

In [29]:
try:
    response = httpx.get(f"{BASE_URL}/products/1", timeout=5.0)
    response.raise_for_status()
    product = response.json()
    print("Product:", product["title"])

except httpx.TimeoutException:
    print("The request took too long. Try again later.")

except httpx.HTTPStatusError as error:
    print("The server returned an error status:", error.response.status_code)

except httpx.RequestError as error:
    print("Could not complete the request:", error)

Product: Essence Mascara Lash Princess


A **timeout** is a limit on how long the client waits. It prevents your program from waiting indefinitely. A timeout does not necessarily mean the server did not process the request; be careful about automatically retrying operations that create or charge for something.

In [30]:
import time

# DummyJSON lets us add an artificial delay so we can practise with slow responses.
#   params={'delay': 500}  ->  becomes ?delay=500, telling the server 'wait 500 milliseconds
#                              (half a second) before replying'. Great for feeling real latency.
#   timeout=15.0           ->  we raise the timeout to 15s so our own limit doesn't fire first.
start = time.perf_counter()              # perf_counter() = a precise stopwatch (in seconds)
r = httpx.get(f'{BASE_URL}/products/1', params={'delay': 500}, timeout=15.0)
print('Got:', r.json()['title'])
print(f'One slow call took {time.perf_counter()-start:.1f}s')   # subtract start = elapsed seconds

Got: Essence Mascara Lash Princess
One slow call took 1.4s


In [31]:
# Now make FIVE slow calls one after another. Each waits ~0.5s, and because we do them
# one-by-one (each finishes before the next starts), the times ADD UP to about 2.5 seconds.
start = time.perf_counter()
for i in range(5):                       # i goes 0,1,2,3,4
    httpx.get(f'{BASE_URL}/products/{i+1}', params={'delay': 500}, timeout=15.0)
print(f'Five slow calls, one after another, took {time.perf_counter()-start:.1f}s')

Five slow calls, one after another, took 3.8s


## 9. Connect an API response to Pydantic

HTTP gives us data; Pydantic can validate that data before our application uses it. This joins the concepts from Day 08.

In [32]:
from pydantic import BaseModel, Field

class ProductSummary(BaseModel):
    id: int = Field(gt=0)
    title: str
    price: float = Field(ge=0)
    xyz:str | None = None # optional field

response = httpx.get(f"{BASE_URL}/products/1", timeout=10.0)
response.raise_for_status()

raw_product = response.json()
product = ProductSummary.model_validate(raw_product)

print("ID:", product.id)
print("Title:", product.title)
print("Price:", product.price)

ID: 1
Title: Essence Mascara Lash Princess
Price: 9.99


The response can contain many additional fields. This small model validates the fields our application needs.

**Pattern:** `HTTP request → response JSON → Pydantic validation → application logic`

## Summary

| Concept | Example |
|---|---|
| Ask a real server for data | `httpx.get('https://dummyjson.com/products')` |
| GET | `httpx.get(url)` |
| POST with JSON | `httpx.post(url, json=data)` |
| PUT / PATCH | Update a resource |
| DELETE | Request resource removal |
| Query parameters | `params={"limit": 5}` |
| Status code | `response.status_code` |
| JSON response | `response.json()` |
| Raise on HTTP error | `response.raise_for_status()` |
| Timeout | `timeout=5.0` |
| Error handling | Catch `httpx` exceptions |
| Validate API data | `Model.model_validate(response.json())` |

**Big idea:** an API call is a request to another program. Your code sends the request, checks the response, and validates the data before relying on it.